In [1]:
import math
import time
import requests
import pandas as pd
import warnings
from pathlib import Path
BASE_URL = "https://www.nseindia.com/api/annual-reports"
HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://www.nseindia.com/",
    "Origin": "https://www.nseindia.com/"
}

excel_path = r"C:\Users\kaustubh.keny\OneDrive - Cogencis Information Services Ltd\Desktop\Important NSE docs\NSE Listed Companies.xlsx"
link_path = r"PDF_LINKS_NSE.xlsx"


df = pd.read_excel(excel_path)
SYMBOLS = df[df["Indices"] != "Rest Of Companies"]["NSE Exchange Symbol"].to_list()
len(SYMBOLS)

750

FETCH ANNUAL REPORT PDF LINKS FROM NSE

In [ ]:

def fetch_category(symbol):

    all_records = []
    params = {
        "index": "equities",
        "symbol": symbol
    }

    r = requests.get(
        BASE_URL,
        params=params,
        headers=HEADERS,
        timeout=30,
        verify=False
    )

    r.raise_for_status()

    data = r.json()
    rows = data.get("data", [])
    all_records.extend(rows)

    return pd.DataFrame(all_records)

all_dfs = []

for idx,cat in enumerate(SYMBOLS):
    try:
        df = fetch_category(cat)

        if not df.empty:
            df["SYMBOL"] = cat
            all_dfs.append(df)

    except Exception as e:
        print(f"Error for {cat}: {e}")

final_df = pd.concat(all_dfs, ignore_index=True)

print("Total records:", len(final_df))

DOWNLOAD THE PDF

In [ ]:
path_download = "Data000.xlsx"
df1 = pd.read_excel(path_download)
df1.columns


In [ ]:
condition = (df1.toYr == 2025) & (df1.submission_type == "New")
nse750 = df1[condition]

session = requests.Session()

session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.nseindia.com/"
})

# Get cookies first
session.get("https://www.nseindia.com", verify=False)


for _, row in nse750.iterrows():
    
    pdf_url = row.fileName
    name = row.SYMBOL
    year = row.toYr
    print(f"DOWNLOAD: {name}..")
    response = session.get(pdf_url, stream=True, verify=False)
    
    response.raise_for_status()
    output_file = f"{year}_{name}.pdf"
    with open(output_file, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    time.sleep(.2)
    print(f"Downloaded: {output_file}")

REMOVE BAD OR CORRUPTED PDFS

In [ ]:
import os
import shutil
import pandas as pd
from PyPDF2 import PdfReader

SOURCE_FOLDER = r"C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\RANDOM_FETCH_DATA\NSE_FETCH\ANNUAL_REPORTS_2026"
BAD_FOLDER = os.path.join(SOURCE_FOLDER, "_BAD_PDFS")

os.makedirs(BAD_FOLDER, exist_ok=True)

bad_files = []

for file in os.listdir(SOURCE_FOLDER):

    if not file.lower().endswith(".pdf"):
        continue

    pdf_path = os.path.join(SOURCE_FOLDER, file)

    try:
        reader = PdfReader(pdf_path)
        _ = len(reader.pages)  # force read

    except Exception as e:

        bad_files.append({
            "pdf_name": file,
            "pdf_path": pdf_path,
            "error": str(e)
        })

        shutil.move(
            pdf_path,
            os.path.join(BAD_FOLDER, file)
        )

        print(f"Moved: {file}")

# save log
if bad_files:
    pd.DataFrame(bad_files).to_excel(
        os.path.join(SOURCE_FOLDER, "bad_pdf_log.xlsx"),
        index=False
    )

print(f"Problematic PDFs found: {len(bad_files)}")

CREATE PAGE INDEX

In [ ]:
import os
import pandas as pd
from PyPDF2 import PdfReader
from openpyxl import load_workbook
pdf_folder = r"C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\RANDOM_FETCH_DATA\NSE_FETCH\ANNUAL_REPORTS_2026"
output_excel = "pdf_page_index2.xlsx"

rows = []
for file_name in os.listdir(pdf_folder):
    if file_name.lower().endswith(".pdf"):
        pdf_path = os.path.join(pdf_folder, file_name)

        try:
            reader = PdfReader(pdf_path)
            page_count = len(reader.pages)
            for page_num in range(1, page_count + 1):
                rows.append(
                    {
                        "pdf_name": file_name,
                        "page_number": page_num,
                      
                    }
                )

        except Exception as e:
            print(f"Error processing {file_name}: {e}")


df = pd.DataFrame(rows)
with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="PDF Index", index=False)

print(f"Created: {output_excel}")

Created: pdf_page_index2.xlsx
